In [ ]:
!pip install -q transformers datasets pytorch-lightning jiwer gdown model2vec

import os
import zipfile
import gdown
from datasets import load_dataset, Audio, DatasetDict

import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperProcessor

import pytorch_lightning as pl
from transformers import WhisperForConditionalGeneration
import jiwer

import json

from torch.utils.data import DataLoader

from pytorch_lightning.callbacks import ModelCheckpoint
import pytorch_lightning as pl

In [ ]:
file_id = '1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'toronto_dataset.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall('toronto_data')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.0 MB/s eta 0:00:00


Downloading...
From (original): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ
From (redirected): https://drive.google.com/uc?id=1j9d91QqE7_WnOnmEmidtOG55tpmxQUeJ&confirm=t&uuid=c1c547e5-23cf-4917-b19f-9d91faba9e47
To: /content/toronto_dataset.zip
100%|██████████| 9.12G/9.12G [01:51<00:00, 81.9MB/s]


In [ ]:
test_ids = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

raw_dataset = load_dataset("audiofolder", data_dir="toronto_data")

raw_dataset = raw_dataset.cast_column("audio", Audio(decode=False))

def is_test(example):
    file_name = os.path.basename(example['audio']['path']).split('.')[0]
    return file_name in test_ids

test_ds = raw_dataset['train'].filter(is_test)
train_val_ds = raw_dataset['train'].filter(lambda x: not is_test(x))

test_ds = test_ds.cast_column("audio", Audio(decode=True))
train_val_ds = train_val_ds.cast_column("audio", Audio(decode=True))

train_val_split = train_val_ds.train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train": train_val_split["train"],
    "validation": train_val_split["test"],
    "test": test_ds
})

print(f"Dataset ready: {dataset}")

Resolving data files:   0%|          | 0/18304 [00:00<?, ?it/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Dataset ready: DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 16472
    })
    validation: Dataset({
        features: ['audio', 'label'],
        num_rows: 1831
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 0
    })
})


In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="uk", task="transcribe")

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [15]:
class WhisperLightningModule(pl.LightningModule):
    def __init__(self, model_name_or_path="openai/whisper-tiny", learning_rate=1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)

        self.model.generation_config.forced_decoder_ids = None
        self.model.generation_config.suppress_tokens = []

        self.model.config.use_cache = False

    def forward(self, input_features, labels=None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        outputs = self(input_features=input_features, labels=labels)
        val_loss = outputs.loss
        self.log("val_loss", val_loss, prog_bar=True, sync_dist=True)

        predicted_ids = self.model.generate(input_features)

        labels[labels == -100] = processor.tokenizer.pad_token_id

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
        reference = processor.batch_decode(labels, skip_special_tokens=True)

        try:
            wer = jiwer.wer(reference, transcription)
            cer = jiwer.cer(reference, transcription)
            self.log("val_wer", wer, prog_bar=True, sync_dist=True)
            self.log("val_cer", cer, prog_bar=True, sync_dist=True)
        except ValueError:
            pass

        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer

In [ ]:
print("Доступні колонки:", dataset["train"].column_names)
print("Приклад даних:", dataset["train"][0])

Доступні колонки: ['audio', 'label']
Приклад даних: {'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7da8f5f82330>, 'label': 64}


In [12]:
test_ids = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

transcripts = {}
with open("toronto_data/labels.jsonl", "r", encoding="utf-8") as f:
    data = json.loads(f.readline())
    for path, text in data.items():
        file_name = os.path.basename(path)
        transcripts[file_name] = text

raw_dataset = load_dataset("audiofolder", data_dir="toronto_data")
raw_dataset = raw_dataset.cast_column("audio", Audio(decode=False))

def is_test_fixed(example):
    file_name = os.path.basename(example["audio"]["path"])

    prefix = "_".join(file_name.split("_")[:2])
    return prefix in test_ids

test_ds = raw_dataset['train'].filter(is_test_fixed)
train_val_ds = raw_dataset['train'].filter(lambda x: not is_test_fixed(x))

train_val_split = train_val_ds.train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train": train_val_split["train"],
    "validation": train_val_split["test"],
    "test": test_ds
})

print(f"Розмір train: {len(dataset['train'])}")
print(f"Розмір val: {len(dataset['validation'])}")
print(f"Розмір test: {len(dataset['test'])}")

def add_text_to_dataset(example):
    file_name = os.path.basename(example["audio"]["path"])
    example["sentence"] = transcripts.get(file_name, "")
    return example

dataset = dataset.map(add_text_to_dataset)
dataset = dataset.filter(lambda x: x["sentence"] != "")

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000, decode=True))

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("Починаємо фінальний препроцесинг...")
encoded_dataset = dataset.map(prepare_dataset, remove_columns=["audio", "label", "sentence"], num_proc=2)
print("Готово!")

Resolving data files:   0%|          | 0/18304 [00:00<?, ?it/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Розмір train: 11484
Розмір val: 1277
Розмір test: 5542


Map:   0%|          | 0/11484 [00:00<?, ? examples/s]

...

Починаємо фінальний препроцесинг...


Map (num_proc=2):   0%|          | 0/11417 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1272 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5521 [00:00<?, ? examples/s]

Готово!


In [13]:
batch_size = 8

train_loader = DataLoader(encoded_dataset["train"], batch_size=batch_size, collate_fn=data_collator, shuffle=True, num_workers=2)
val_loader = DataLoader(encoded_dataset["validation"], batch_size=batch_size, collate_fn=data_collator, num_workers=2)
test_loader = DataLoader(encoded_dataset["test"], batch_size=batch_size, collate_fn=data_collator, num_workers=2)

In [16]:
model = WhisperLightningModule(model_name_or_path="openai/whisper-tiny", learning_rate=1e-5)

checkpoint_callback = ModelCheckpoint(
    dirpath="./whisper_checkpoints",
    filename="whisper-tiny-best-{epoch:02d}-{val_wer:.2f}",
    save_top_k=1,
    monitor="val_wer",
    mode="min"
)

trainer = pl.Trainer(
    max_epochs=3,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback],
    log_every_n_steps=10,
    precision="16-mixed"
)

print("Починаємо тренування...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Починаємо тренування...


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │ 37.8 M │ eval │     0 │
└───┴───────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 37.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 37.8 M                                                                                               
Total estimated model params size (MB): 151                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 126                                                                                          
Total FLOPs: 0

Output()

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 126 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.


In [17]:
best_model_path = checkpoint_callback.best_model_path
print(f"Найкраща модель: {best_model_path}")

best_model = WhisperLightningModule.load_from_checkpoint(best_model_path)

test_results = trainer.validate(best_model, dataloaders=test_loader)

print("\Результати на тестовій вибірці:")
print(test_results)

Найкраща модель: /content/whisper_checkpoints/whisper-tiny-best-epoch=00-val_wer=4.73.ckpt


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_cer          │    3.8936386108398438     │
│         val_loss          │     0.894510805606842     │
│          val_wer          │     4.809018135070801     │
└───────────────────────────┴───────────────────────────┘

\Результати на тестовій вибірці:
[{'val_loss': 0.894510805606842, 'val_wer': 4.809018135070801, 'val_cer': 3.8936386108398438}]
